# Formal verification of Q# circuits

Below is a prototype of formal verification of addition circuit in Q# using z3. It is based on my [VeriCirq](https://github.com/fedimser/vericirq) project.

It currently supports only X, CNOT and CCNOT gates. 

In [1]:
import qdk
import json
import z3

def _bit_to_bool(bitvec_var: z3.BitVecRef, bit_id: int) -> z3.BoolRef:
    return z3.Extract(bit_id, bit_id, bitvec_var) == 1

class QsharpCircuitVerifier:

    def __init__(self, circuit, input_sizes):
        ct = json.loads(circuit.json())
 
        self.solver = z3.Solver()

        num_qubits = len(ct['qubits'])
        num_input_qubits = sum(input_sizes)
        num_ancilla = num_qubits - num_input_qubits
        self.input_qubits = [z3.FreshBool() for _ in range(num_input_qubits)] 
        self.qubits = self.input_qubits + [z3.BoolVal(False) for _ in range(num_ancilla)]
        
        self.dfs(ct['componentGrid'])

        self.input_vars = []
        self.output_vars = []
        offset=0
        for i, input_size in enumerate(input_sizes):
            input_var = z3.BitVec(f"in_{i}", input_size)
            self.input_vars.append(input_var)
            for j in range(input_size):
                self.solver.add(self.input_qubits[offset+j] == _bit_to_bool(input_var, j))
            output_var = z3.BitVec(f"out_{i}", input_size)
            self.output_vars.append(output_var)
            for j in range(input_size):
                self.solver.add(self.qubits[offset+j] == _bit_to_bool(output_var, j))
            offset += input_size
        
    def apply_x(self, target):
        self.qubits[target] = z3.Not(self.qubits[target])

    def apply_cx(self, control, target):
        output = z3.FreshBool()
        self.solver.add(output == z3.Xor(self.qubits[control], self.qubits[target]))
        self.qubits[target] = output

    def apply_ccx(self, control1, control2, target):
        output = z3.FreshBool()
        self.solver.add(output == z3.Xor(z3.And(self.qubits[control1], self.qubits[control2]), self.qubits[target]))
        self.qubits[target] = output
    
    def dfs(self, node: dict):
        if type(node) is list:
            for child in node:
                self.dfs(child)
        elif 'components' in node:
            self.dfs(node['components'])
        elif 'children' in node:
            self.dfs(node['children'])
        elif 'gate' in node and node['gate'] == "X":
            assert len(node['targets']) == 1
            target = node['targets'][0]['qubit']
            controls = node['controls']
            assert 0 <= len(controls) <=2
            if len(controls)==0:
                self.apply_x(target)
            elif len(controls)==1:
                self.apply_cx(controls[0]['qubit'], target)
            elif len(controls)==2:
                self.apply_ccx(controls[0]['qubit'], controls[1]['qubit'], target)
        else:
            raise RuntimeError("Cannot process gate:", node)

    def verify_spec(self, spec: z3.BoolRef):
        """Verifies user-specified condition about inputs and outputs.

        Spec must be a z3 expression that states a fact about input and output variables.
        The variables can be accessed using `input_vars` and `output_vars` fields on `GateVerifier`.
        """
        self.solver.push()
        self.solver.add(z3.Not(spec))

        if self.solver.check() == z3.sat:
            model = self.solver.model()
            inputs = [model.evaluate(v).as_long() for v in self.input_vars]
            outputs = [model.evaluate(v).as_long() for v in self.output_vars]
            print("FAIL", inputs, outputs)
        else:
            print("OK")

        self.solver.pop()


ctx = qdk.Context()
ct = ctx.circuit("""{
  use x1 = Qubit[5];
  use x2 = Qubit[5]; 
  Std.Arithmetic.RippleCarryTTKIncByLE(x1,x2);
}""")
ver = QsharpCircuitVerifier(ct, input_sizes=[5,5])
x_in,y_in = ver.input_vars
x_out, y_out = ver.output_vars
ver.verify_spec(x_in==x_out)
ver.verify_spec(y_out==x_in+y_in)


OK
OK
